# PagedAttention 教程

本教程介绍 PagedAttention 的核心概念和实现，这是 vLLM 等高效推理框架的关键技术。

## 目录
1. KV Cache 内存问题
2. PagedAttention 原理
3. 块分配器实现
4. Copy-on-Write 优化
5. 实战演示

## 1. KV Cache 内存问题

### 传统 KV Cache 的问题

在 Transformer 推理中，每个 token 需要存储 Key 和 Value：

$$\text{KV Cache Size} = 2 \times L \times H \times D \times S \times B$$

其中：
- $L$: 层数
- $H$: 注意力头数
- $D$: 头维度
- $S$: 序列长度
- $B$: 批大小

**问题**：
1. 预分配最大长度导致内存浪费
2. 不同请求长度不同，内存碎片严重
3. 无法动态调整批大小

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from src.paged_attention import (
    PagedAttentionConfig,
    BlockAllocator,
    PagedKVCache,
    PagedAttention,
    create_paged_attention
)

# 计算传统 KV Cache 内存
def calc_traditional_kv_cache_memory(
    num_layers: int,
    num_heads: int,
    head_dim: int,
    max_seq_len: int,
    batch_size: int,
    dtype_bytes: int = 2  # FP16
) -> float:
    """计算传统 KV Cache 内存 (GB)"""
    kv_size = 2 * num_layers * num_heads * head_dim * max_seq_len * batch_size * dtype_bytes
    return kv_size / (1024 ** 3)

# LLaMA-7B 参数
memory_gb = calc_traditional_kv_cache_memory(
    num_layers=32,
    num_heads=32,
    head_dim=128,
    max_seq_len=2048,
    batch_size=32
)
print(f"LLaMA-7B KV Cache (batch=32, seq=2048): {memory_gb:.2f} GB")

## 2. PagedAttention 原理

PagedAttention 借鉴操作系统的虚拟内存分页机制：

1. **物理块 (Physical Block)**: 固定大小的 KV Cache 存储单元
2. **逻辑块 (Logical Block)**: 序列视角的块索引
3. **块表 (Block Table)**: 逻辑块到物理块的映射

```
Sequence 1: [Block 0] -> [Block 3] -> [Block 7]
Sequence 2: [Block 1] -> [Block 5]
Sequence 3: [Block 2] -> [Block 4] -> [Block 6]
```

**优势**：
- 按需分配，减少内存浪费
- 支持动态序列长度
- 便于实现 Copy-on-Write

In [ ]:
# 创建 PagedAttention 配置
config = PagedAttentionConfig(
    block_size=16,      # 每块存储 16 个 token
    num_blocks=100,     # 总共 100 个物理块
    num_heads=8,
    head_dim=64,
    num_layers=4
)

print(f"块大小: {config.block_size} tokens")
print(f"总块数: {config.num_blocks}")
print(f"每块内存: {config.block_memory_bytes / 1024:.2f} KB")
print(f"总内存: {config.total_memory_bytes / (1024**2):.2f} MB")

## 3. 块分配器实现

块分配器管理物理块的分配和释放。

In [ ]:
# 创建块分配器
allocator = BlockAllocator(
    num_blocks=20,
    block_size=16,
    num_heads=8,
    head_dim=64
)

print(f"初始空闲块: {allocator.get_num_free_blocks()}")

# 分配块
block1 = allocator.allocate()
block2 = allocator.allocate()
block3 = allocator.allocate()

print(f"分配 3 块后空闲: {allocator.get_num_free_blocks()}")
print(f"已分配块 ID: {block1.block_id}, {block2.block_id}, {block3.block_id}")

# 释放块
allocator.free(block1.block_id)
print(f"释放 1 块后空闲: {allocator.get_num_free_blocks()}")

## 4. Copy-on-Write 优化

当多个序列共享相同前缀时（如 beam search），使用 Copy-on-Write 避免重复存储。

In [ ]:
# 创建 PagedKVCache
kv_cache = PagedKVCache(config)

# 分配序列
seq1 = kv_cache.allocate_sequence()
print(f"分配序列 1: ID={seq1}")

# 添加一些 token
for i in range(20):  # 20 tokens，需要 2 个块
    k = np.random.randn(1, config.num_heads, config.head_dim).astype(np.float32)
    v = np.random.randn(1, config.num_heads, config.head_dim).astype(np.float32)
    kv_cache.append_tokens(seq1, k, v, layer_idx=0)

print(f"序列 1 添加 20 tokens 后")

# Fork 序列 (Copy-on-Write)
seq2 = kv_cache.fork_sequence(seq1)
print(f"Fork 序列 2: ID={seq2}")
print(f"当前序列数: {kv_cache.get_num_sequences()}")

## 5. 实战演示

使用 PagedAttention 进行推理。

In [ ]:
# 创建 PagedAttention
paged_attn = create_paged_attention(
    num_heads=8,
    head_dim=64,
    num_layers=2,
    block_size=16,
    num_blocks=50
)

# Prefill 阶段
batch_size = 2
seq_len = 32
query = np.random.randn(batch_size, seq_len, 8, 64).astype(np.float32)
key = np.random.randn(batch_size, seq_len, 8, 64).astype(np.float32)
value = np.random.randn(batch_size, seq_len, 8, 64).astype(np.float32)

output, seq_ids = paged_attn.forward(
    query, key, value,
    layer_idx=0,
    is_prefill=True
)

print(f"Prefill 输出形状: {output.shape}")
print(f"分配的序列 ID: {seq_ids}")
print(f"内存使用: {paged_attn.get_memory_usage():.2f} MB")

In [ ]:
# Decode 阶段
for step in range(5):
    # 每步生成一个 token
    q = np.random.randn(batch_size, 1, 8, 64).astype(np.float32)
    k = np.random.randn(batch_size, 1, 8, 64).astype(np.float32)
    v = np.random.randn(batch_size, 1, 8, 64).astype(np.float32)
    
    output, _ = paged_attn.forward(
        q, k, v,
        layer_idx=0,
        is_prefill=False,
        sequence_ids=seq_ids
    )
    
print(f"Decode 5 步后内存: {paged_attn.get_memory_usage():.2f} MB")

## 总结

PagedAttention 的核心优势：

1. **内存效率**: 按需分配，减少 60-80% 内存浪费
2. **动态批处理**: 支持不同长度序列混合
3. **Copy-on-Write**: 高效支持 beam search
4. **可扩展性**: 更大批大小，更高吞吐量

### 参考资料
- [vLLM Paper](https://arxiv.org/abs/2309.06180)
- [PagedAttention Blog](https://blog.vllm.ai/2023/06/20/vllm.html)